In [83]:
import pandas as pd
from IPython.display import display
import polars as pl

In [84]:
# Téléchargement des dataframes depuis la base de données des BAAC.
urls_y = {
    2022: {
        "carac" : "https://www.data.gouv.fr/api/1/datasets/r/5fc299c0-4598-4c29-b74c-6a67b0cc27e7",
        "lieux" : "https://www.data.gouv.fr/api/1/datasets/r/a6ef711a-1f03-44cb-921a-0ce8ec975995",
        "usagers" : "https://www.data.gouv.fr/api/1/datasets/r/62c20524-d442-46f5-bfd8-982c59763ec8"
    },
    2023 : {
        "carac" : "https://www.data.gouv.fr/api/1/datasets/r/104dbb32-704f-4e99-a71e-43563cb604f2",
        "lieux" : "https://www.data.gouv.fr/api/1/datasets/r/8bef19bf-a5e4-46b3-b5f9-a145da4686bc",
        "usagers" : "https://www.data.gouv.fr/api/1/datasets/r/68848e2a-28dd-4efc-9d5f-d512f7dbe66f"
    },
    2024 : {
        "carac" : "https://www.data.gouv.fr/api/1/datasets/r/83f0fb0e-e0ef-47fe-93dd-9aaee851674a",
        "lieux" : "https://www.data.gouv.fr/api/1/datasets/r/228b3cda-fdfb-4677-bd54-ab2107028d2d",
        "usagers" : "https://www.data.gouv.fr/api/1/datasets/r/f57b1f58-386d-4048-8f78-2ebe435df868"
    }
}

url_05_21 = {
    "carac" : "https://www.data.gouv.fr/api/1/datasets/r/a3cac8bc-4a07-4124-8a08-633a3a91d40b",
    "lieux" : "https://www.data.gouv.fr/api/1/datasets/r/b7f25e45-de32-4801-b0eb-62989f1a7406",
    "usagers" : "https://www.data.gouv.fr/api/1/datasets/r/a64b1b9f-4d56-4b26-ae90-9f40b878e109"
}

# Dictionnaire qui contiendra les dataframes pour chaque année en vue de la concaténation

dfs = {}

yrl = [2022,2023,2024]
names = ["carac","lieux","usagers"]

for name in names:
    list_df = []
    for yr in yrl:
        url = urls_y[yr][name]
        df_yr = pd.read_csv(url,sep=";") # Le séparateur utilisé est ";"
        df_yr.insert(1,"year",yr) # Colonne année pour distinguer les accidents entre années en vue des opérations de fusion de dataframes
        list_df.append(df_yr)

    concatenated_df_type = pd.concat(list_df,ignore_index=True) # On fait fi de l'index
    dfs[name] = concatenated_df_type # On associe le dataframe des trois années au type de dataframe

dfs


C:\Users\Lucy Neveux\AppData\Local\Temp\ipykernel_67436\2870287309.py:37: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_yr = pd.read_csv(url,sep=";") # Le séparateur utilisé est ";"
C:\Users\Lucy Neveux\AppData\Local\Temp\ipykernel_67436\2870287309.py:37: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df_yr = pd.read_csv(url,sep=";") # Le séparateur utilisé est ";"


{'carac':          Accident_Id  year  jour  mois    an   hrmn  lum dep    com  agg  int  \
 0       2.022000e+11  2022    19    10  2022  16:15    1  26  26198    2    3   
 1       2.022000e+11  2022    20    10  2022  08:34    1  25  25204    2    3   
 2       2.022000e+11  2022    20    10  2022  17:15    1  22  22360    2    6   
 3       2.022000e+11  2022    20    10  2022  18:00    1  16  16102    2    3   
 4       2.022000e+11  2022    19    10  2022  11:45    1  13  13103    1    1   
 ...              ...   ...   ...   ...   ...    ...  ...  ..    ...  ...  ...   
 164521           NaN  2024    10     7  2024  02:05    5  94  94065    2    3   
 164522           NaN  2024    30    11  2024  15:27    1  92  92062    2    1   
 164523           NaN  2024    24    10  2024  20:40    3  29  29068    1    1   
 164524           NaN  2024    30    11  2024  15:40    1  92  92012    2    1   
 164525           NaN  2024    10     7  2024  08:30    1  94  94028    2    1   
 
     

In [85]:
if "an" in dfs["carac"].columns:
    dfs["carac"].drop(columns=["an"],inplace=True) # Colonne an redondante avec colonne year

Observons la longueur de chacun des dataframes :

In [86]:
for n,content in dfs.items(): 
    print(n,len(content))

carac 164526
lieux 196410
usagers 377638


Il est normal que le dataframe "usagers" soit le plus long : en effet, pour chaque accident corporel, on peut compter plusieurs victimes. Il est plus surprenant que "lieux" soit plus long que "carac". 
Observons les doublons :

In [87]:
dfs["lieux"]["Num_Acc"].value_counts()

Num_Acc
202300035508    5
202400050330    5
202300039940    4
202300034748    4
202300006227    4
               ..
202200000042    1
202200000043    1
202200000044    1
202200000045    1
202200000006    1
Name: count, Length: 164526, dtype: int64

In [88]:
dfs["lieux"][dfs["lieux"]["Num_Acc"]==202300035508]

,Num_Acc,year,catr,voie,v1,v2,circ,nbv,vosp,prof,pr,pr1,plan,lartpc,larrout,surf,infra,situ,vma
101231,202300035508,2023,4,BOULEVARD DE BEAUSEJOUR,0,NaN,1,0,0,1,0,0,1,NaN,-1,1,0,1,30
101232,202300035508,2023,4,BOULEVARD EMILE AUGIER,0,NaN,3,2,0,1,-1,-1,1,NaN,-1,1,9,1,50
101233,202300035508,2023,4,CHAUSSEE LA MUETTE,0,NaN,3,1,3,1,-1,-1,1,NaN,-1,1,6,1,50
101234,202300035508,2023,4,RUE D ANDIGNE,0,NaN,1,1,0,1,0,0,1,NaN,-1,1,0,1,30
101235,202300035508,2023,4,RUE LARGILLIERE,0,NaN,2,2,0,1,0,0,1,NaN,-1,1,0,1,30


In [89]:
sum(dfs["carac"]["Num_Acc"].value_counts()>1)

0

In [90]:
for name in names:
    print(name,dfs[name].columns)

carac Index(['Accident_Id', 'year', 'jour', 'mois', 'hrmn', 'lum', 'dep', 'com',
       'agg', 'int', 'atm', 'col', 'adr', 'lat', 'long', 'Num_Acc'],
      dtype='object')
lieux Index(['Num_Acc', 'year', 'catr', 'voie', 'v1', 'v2', 'circ', 'nbv', 'vosp',
       'prof', 'pr', 'pr1', 'plan', 'lartpc', 'larrout', 'surf', 'infra',
       'situ', 'vma'],
      dtype='object')
usagers Index(['Num_Acc', 'year', 'id_usager', 'id_vehicule', 'num_veh', 'place',
       'catu', 'grav', 'sexe', 'an_nais', 'trajet', 'secu1', 'secu2', 'secu3',
       'locp', 'actp', 'etatp'],
      dtype='object')


In [91]:
dfs["usagers"]

,Num_Acc,year,id_usager,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
0,202200000001,2022,1 099 700,813 952,A01,1,1,3,1,2008.0,5,2,8,-1,-1,-1,-1
1,202200000001,2022,1 099 701,813 953,B01,1,1,1,1,1948.0,5,1,8,-1,-1,-1,-1
2,202200000002,2022,1 099 698,813 950,B01,1,1,4,1,1988.0,9,1,0,-1,0,0,-1
3,202200000002,2022,1 099 699,813 951,A01,1,1,1,1,1970.0,4,1,0,-1,0,0,-1
4,202200000003,2022,1 099 696,813 948,A01,1,1,1,1,2002.0,0,1,0,-1,-1,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
377633,202400054401,2024,203 859 570,155 686 119,Y01,1,1,4,2,1978.0,0,0,0,0,-1,-1,-1
377634,202400054401,2024,203 859 572,155 686 120,A01,1,1,1,1,1984.0,0,2,6,0,-1,-1,-1
377635,202400054402,2024,203 859 569,155 686 118,A01,1,1,4,1,1981.0,4,1,0,-1,-1,-1,-1
377636,202400054402,2024,203 859 571,155 686 121,B01,1,1,4,2,1986.0,9,1,0,-1,-1,-1,-1


"Lieux" indique les différentes voies liées à l'accident lorsqu'il a eu lieu dans une intersection complexe. Il contient par ailleurs différentes informations sur le lieu de l'accident (type de route **catr**, l'état de la surface **surf**, la vitesse maximale autorisée **vma**).
Nous le laissons de côté pour l'instant.

Créons maintenant un premier dataset, qui fusionne les données de 2022,2023, et 2024 pour les dataframes **usagers** et **carac** :

In [92]:
if "Accident_Id" in dfs["carac"].columns:
    dfs["carac"]["Num_Acc"] = dfs["carac"]["Num_Acc"].fillna(dfs["carac"]["Accident_Id"])   # On fusionne Accident_Id et Num_Acc, qui représentent le même indicateur
    dfs["carac"].drop(columns=["Accident_Id"],inplace=True)

col_data = dfs["carac"].pop("Num_Acc")
dfs["carac"].insert(0,"Num_Acc",col_data) # On replace la colonne "Num_Acc" en index 0 des colonnes du dataframe

In [93]:
dfs["carac"]

,Num_Acc,year,jour,mois,hrmn,lum,dep,com,agg,int,atm,col,adr,lat,long
0,2.022000e+11,2022,19,10,16:15,1,26,26198,2,3,1,3,TEIL(vieille route du),"44,5594200000","4,7257200000"
1,2.022000e+11,2022,20,10,08:34,1,25,25204,2,3,1,3,Miranda,"46,9258100000","6,3462000000"
2,2.022000e+11,2022,20,10,17:15,1,22,22360,2,6,1,2,ROND POINT DE BREZILLET,"48,4931620000","-2,7604390000"
3,2.022000e+11,2022,20,10,18:00,1,16,16102,2,3,8,6,LOHMEYER (RUE),"45,6926520000","-0,3262900000"
4,2.022000e+11,2022,19,10,11:45,1,13,13103,1,1,1,2,ROUTE DE JEAN MOULIN-RN 538,"43,6755790366","5,0927031775"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164521,2.024001e+11,2024,10,7,02:05,5,94,94065,2,3,2,6,Avenue Charles Lindbergh,"48,75740000","2,34469000"
164522,2.024001e+11,2024,30,11,15:27,1,92,92062,2,1,1,6,RUE PIERRE GAUDIN,"48,88694322","2,24863619"
164523,2.024001e+11,2024,24,10,20:40,3,29,29068,1,1,1,1,PK 031.950 RN12,"48,52619024","-3,96317650"
164524,2.024001e+11,2024,30,11,15:40,1,92,92012,2,1,1,3,JEAN JAURES (BOULEVARD) 63/215 - 70/208,"48,83286197","2,24394009"


In [94]:
KEY = ["year","Num_Acc"]

df_final2224 = dfs["usagers"].merge(dfs["carac"],on=KEY,how="left") # On garde toutes les colonnes usagers

print(df_final2224.shape[0]==dfs["usagers"].shape[0])

# df_final et dfs["usagers"] font bien la même longueur.

True


In [95]:
sum(df_final2224["grav"].isna())

0

In [96]:
df_final2224

,Num_Acc,year,id_usager,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,...,lum,dep,com,agg,int,atm,col,adr,lat,long
0,202200000001,2022,1 099 700,813 952,A01,1,1,3,1,2008.0,...,1,26,26198,2,3,1,3,TEIL(vieille route du),"44,5594200000","4,7257200000"
1,202200000001,2022,1 099 701,813 953,B01,1,1,1,1,1948.0,...,1,26,26198,2,3,1,3,TEIL(vieille route du),"44,5594200000","4,7257200000"
2,202200000002,2022,1 099 698,813 950,B01,1,1,4,1,1988.0,...,1,25,25204,2,3,1,3,Miranda,"46,9258100000","6,3462000000"
3,202200000002,2022,1 099 699,813 951,A01,1,1,1,1,1970.0,...,1,25,25204,2,3,1,3,Miranda,"46,9258100000","6,3462000000"
4,202200000003,2022,1 099 696,813 948,A01,1,1,1,1,2002.0,...,1,22,22360,2,6,1,2,ROND POINT DE BREZILLET,"48,4931620000","-2,7604390000"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
377633,202400054401,2024,203 859 570,155 686 119,Y01,1,1,4,2,1978.0,...,1,92,92012,2,1,1,3,JEAN JAURES (BOULEVARD) 63/215 - 70/208,"48,83286197","2,24394009"
377634,202400054401,2024,203 859 572,155 686 120,A01,1,1,1,1,1984.0,...,1,92,92012,2,1,1,3,JEAN JAURES (BOULEVARD) 63/215 - 70/208,"48,83286197","2,24394009"
377635,202400054402,2024,203 859 569,155 686 118,A01,1,1,4,1,1981.0,...,1,94,94028,2,1,1,4,Avenue de la Pompadour,"48,78330000","2,46660000"
377636,202400054402,2024,203 859 571,155 686 121,B01,1,1,4,2,1986.0,...,1,94,94028,2,1,1,4,Avenue de la Pompadour,"48,78330000","2,46660000"


La gravité des blessures de chaque victime est renseignée, ce qui nous évite un travail supplémentaire de preprocessing pour la variable d'intérêt **grav**.

>Pour rappel, **grav** peut prendre quatre valeurs : 
>* 1 = Indemne
>* 2 = Tué
>* 3 = Blessé hospitalisé
>* 4 = Blessé léger
    

In [97]:
url_05_21.items()

dict_items([('carac', 'https://www.data.gouv.fr/api/1/datasets/r/a3cac8bc-4a07-4124-8a08-633a3a91d40b'), ('lieux', 'https://www.data.gouv.fr/api/1/datasets/r/b7f25e45-de32-4801-b0eb-62989f1a7406'), ('usagers', 'https://www.data.gouv.fr/api/1/datasets/r/a64b1b9f-4d56-4b26-ae90-9f40b878e109')])

Nous construisons maintenant le dataset contenant les données de 2005 à 2021 en vue d'une harmonisation

In [98]:
df0521={}
for type,link in url_05_21.items():
    df0521[type] = pd.read_csv(link,
                               encoding="latin-1",  # Old files, with a different encoding than the recent ones
                               sep=",")
df0521

C:\Users\Lucy Neveux\AppData\Local\Temp\ipykernel_67436\3103815293.py:3: DtypeWarning: Columns (4,10,13,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df0521[type] = pd.read_csv(link,
C:\Users\Lucy Neveux\AppData\Local\Temp\ipykernel_67436\3103815293.py:3: DtypeWarning: Columns (3,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df0521[type] = pd.read_csv(link,
C:\Users\Lucy Neveux\AppData\Local\Temp\ipykernel_67436\3103815293.py:3: DtypeWarning: Columns (9,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df0521[type] = pd.read_csv(link,


{'carac':          Unnamed: 0       num_acc  mois  jour   hrmn  lum  agg  int  atm  col  \
 0                 1  200500000001     1    12   1900    3    2    1  1.0  3.0   
 1                 2  200500000002     1    21   1600    1    2    1  1.0  1.0   
 2                 3  200500000003     1    21   1845    3    1    1  2.0  1.0   
 3                 4  200500000004     1     4   1615    1    1    1  1.0  5.0   
 4                 5  200500000005     1    10   1945    3    1    1  3.0  6.0   
 ...             ...           ...   ...   ...    ...  ...  ...  ...  ...  ...   
 1121566     1121567  202100056514     1     1  06:10    3    1    1  5.0  6.0   
 1121567     1121568  202100056515     1     1  10:20    1    1    1  2.0  6.0   
 1121568     1121569  202100056516     1     1  18:00    3    1    1  2.0  1.0   
 1121569     1121570  202100056517     1     1  10:55    1    1    2  1.0  6.0   
 1121570     1121571  202100056518     1     2  18:00    3    1    1  3.0  1.0   
 
     

On supprime la colonne redondante d'index "Unnamed: 0" et on renomme les colonnes "num_acc" et "annee" pour être en cohérence avec le dataset df_final2224 :

In [99]:
print(df0521["carac"].columns,"\n",dfs["carac"].columns)

Index(['Unnamed: 0', 'num_acc', 'mois', 'jour', 'hrmn', 'lum', 'agg', 'int',
       'atm', 'col', 'com', 'adr', 'gps', 'lat', 'long', 'dep', 'annee'],
      dtype='object') 
 Index(['Num_Acc', 'year', 'jour', 'mois', 'hrmn', 'lum', 'dep', 'com', 'agg',
       'int', 'atm', 'col', 'adr', 'lat', 'long'],
      dtype='object')


In [100]:
print(df0521["usagers"].columns,"\n",dfs["usagers"].columns)

Index(['Unnamed: 0', 'num_acc', 'place', 'catu', 'grav', 'sexe', 'trajet',
       'secu', 'locp', 'actp', 'etatp', 'an_nais', 'num_veh', 'annee',
       'id_vehicule', 'secu1', 'secu2', 'secu3'],
      dtype='object') 
 Index(['Num_Acc', 'year', 'id_usager', 'id_vehicule', 'num_veh', 'place',
       'catu', 'grav', 'sexe', 'an_nais', 'trajet', 'secu1', 'secu2', 'secu3',
       'locp', 'actp', 'etatp'],
      dtype='object')


In [101]:
for type in ["usagers","carac"]:
    df0521[type]=df0521[type].rename(columns={"annee":"year","num_acc":"Num_Acc"})
    if "Unnamed: 0" in df0521[type].columns:
        df0521[type].drop(columns="Unnamed: 0",inplace=True)

df0521


{'carac':               Num_Acc  mois  jour   hrmn  lum  agg  int  atm  col    com  \
 0        200500000001     1    12   1900    3    2    1  1.0  3.0     11   
 1        200500000002     1    21   1600    1    2    1  1.0  1.0     51   
 2        200500000003     1    21   1845    3    1    1  2.0  1.0     51   
 3        200500000004     1     4   1615    1    1    1  1.0  5.0     82   
 4        200500000005     1    10   1945    3    1    1  3.0  6.0    478   
 ...               ...   ...   ...    ...  ...  ...  ...  ...  ...    ...   
 1121566  202100056514     1     1  06:10    3    1    1  5.0  6.0  33021   
 1121567  202100056515     1     1  10:20    1    1    1  2.0  6.0  38405   
 1121568  202100056516     1     1  18:00    3    1    1  2.0  1.0  26064   
 1121569  202100056517     1     1  10:55    1    1    2  1.0  6.0  33003   
 1121570  202100056518     1     2  18:00    3    1    1  3.0  1.0  78423   
 
                                adr gps            lat           

In [102]:
KEY = ["Num_Acc","year"]
df_final0521 = df0521["usagers"].merge(df0521["carac"],on=KEY,how="left")   # On garde toutes les données de "usagers", qui est plus large que "carac".
df_final0521

,Num_Acc,place,catu,grav,sexe,trajet,secu,locp,actp,etatp,...,agg,int,atm,col,com,adr,gps,lat,long,dep
0,2.005000e+11,1.0,1,4,1,1.0,11.0,0.0,0,0.0,...,2.0,1.0,1.0,3.0,11,CD41B,M,5051500.0,294400.0,590
1,2.005000e+11,1.0,1,3,2,3.0,11.0,0.0,0,0.0,...,2.0,1.0,1.0,3.0,11,CD41B,M,5051500.0,294400.0,590
2,2.005000e+11,2.0,2,1,1,0.0,11.0,0.0,0,0.0,...,2.0,1.0,1.0,3.0,11,CD41B,M,5051500.0,294400.0,590
3,2.005000e+11,4.0,2,1,1,0.0,31.0,0.0,0,0.0,...,2.0,1.0,1.0,3.0,11,CD41B,M,5051500.0,294400.0,590
4,2.005000e+11,5.0,2,1,1,0.0,11.0,0.0,0,0.0,...,2.0,1.0,1.0,3.0,11,CD41B,M,5051500.0,294400.0,590
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2509615,2.021001e+11,1.0,1,4,1,0.0,NaN,0.0,0,-1.0,...,1.0,1.0,2.0,1.0,26064,Route dÃ©partementale 538,M,"44,9112100000","5,0196360000",26
2509616,2.021001e+11,1.0,1,4,1,5.0,NaN,0.0,0,-1.0,...,1.0,1.0,2.0,1.0,26064,Route dÃ©partementale 538,M,"44,9112100000","5,0196360000",26
2509617,2.021001e+11,1.0,1,3,1,0.0,NaN,0.0,0,-1.0,...,1.0,2.0,1.0,6.0,33003,Voie rapide Bassens Ambes,M,"44,9542747363","-0,5179211363",33
2509618,2.021001e+11,1.0,1,3,1,3.0,NaN,-1.0,-1,-1.0,...,1.0,1.0,3.0,1.0,78423,VOLTA (AVENUE),M,"48,7966700000","2,0505000000",78


Steps suivants : 

Observons les différences dans les noms de colonnes du format des anciens datasets (2005-2021), et des nouveaux datasets (2022-2024).

In [103]:
cols_recentes = set(df_final2224.columns)
cols_anciennes = set(df_final0521.columns)

unique_recent = cols_recentes-cols_anciennes # Soustraire les sets permet de ne garder que les colonnes uniquement présentes en 2022-2024
unique_ancien = cols_anciennes-cols_recentes # Colonnes uniquement présentes en 2005-2021

print(unique_ancien,unique_recent)
print(cols_anciennes,cols_recentes)

{'secu', 'gps'} {'id_usager'}
{'an_nais', 'secu2', 'year', 'catu', 'hrmn', 'jour', 'place', 'col', 'long', 'com', 'sexe', 'mois', 'adr', 'secu3', 'dep', 'grav', 'num_veh', 'actp', 'secu1', 'trajet', 'atm', 'Num_Acc', 'lat', 'gps', 'locp', 'id_vehicule', 'agg', 'secu', 'int', 'lum', 'etatp'} {'an_nais', 'secu2', 'year', 'catu', 'hrmn', 'jour', 'place', 'col', 'long', 'com', 'sexe', 'mois', 'adr', 'secu3', 'dep', 'grav', 'num_veh', 'actp', 'secu1', 'trajet', 'atm', 'Num_Acc', 'lat', 'locp', 'id_vehicule', 'agg', 'int', 'lum', 'id_usager', 'etatp'}


Comme on dispose déjà de donnée sur l'adresse de l'accident, et en particulier sur sa longitude et sa latitude, l'ancienne colonne GPS (qui ne prend qu'une seule mystérieuse valeur, "M") n'est pas utile. Nous pouvons la supprimer.
De même, la colonne id_usager n'est pas très intéressante. Elle peut apparaître plusieurs fois (si un individu fait plusieurs accidents dans l'année), mais nous pouvons déjà différencier les individus au sein d'un même numéro d'accident. Chaque accidenté de chaque accident est déjà indexé par l'index du dataframe.
Nous pouvons donc supprimer ces deux colonnes.

Enfin, la colonne "secu" correspond de 2005 à 2021 à un code sur deux caractères : 
- Le premier concerne l'existence d'un équipement de sécurité : 
    1 – Ceinture
    2 – Casque
    3 – Dispositif enfants
    4 – Equipement réfléchissant
    9 – Autre 

- Le second concerne l'utilisation de cet équipement de sécurité :
    1 – Oui
    2 – Non
    3 – Non déterminable

Dans les versions plus récentes, il est question de l'existence ET de l'utilisation d'un équipement de sécurité, jusqu'à trois à la fois (secu1,secu2,secu3):
    -1 – Non renseigné  
    0 – Aucun équipement  
    1 – Ceinture  
    2 – Casque  
    3 – Dispositif enfants  
    4 – Gilet réfléchissant  
    5 – Airbag (2RM/3RM)  
    6 – Gants (2RM/3RM)  
    7 – Gants + Airbag (2RM/3RM)  
    8 – Non déterminable  
    9 – Autre

Cette nouvelle nomenclature (secu1,secu2,secu3) qui permet une analyse plus granulaire a été introduite en 2019, avec id_vehicule (identifiant du véhicule, que nous pouvons également abandonner).

Dans les étapes suivantes, nous pouvons donc abandonner "gps", "id_vehicule", "id_usager", et réfléchir à une modification de la colonne "secu" pour harmoniser les données 05-21 avec 22-24.


In [104]:
df_final0521[df_final0521["year"]==2019]["lat"].value_counts()

lat
43,1213200    24
43,7705930    24
48,7900000    24
44,9566600    24
48,7597300    24
              ..
50,5683130     1
43,7327224     1
43,3041150     1
43,3356100     1
48,8959294     1
Name: count, Length: 18571, dtype: int64

In [105]:
dfs["usagers"].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 377638 entries, 0 to 377637
Data columns (total 17 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Num_Acc      377638 non-null  int64  
 1   year         377638 non-null  int64  
 2   id_usager    377638 non-null  object 
 3   id_vehicule  377638 non-null  object 
 4   num_veh      377638 non-null  object 
 5   place        377638 non-null  int64  
 6   catu         377638 non-null  int64  
 7   grav         377638 non-null  int64  
 8   sexe         377638 non-null  int64  
 9   an_nais      369587 non-null  float64
 10  trajet       377638 non-null  int64  
 11  secu1        377638 non-null  int64  
 12  secu2        377638 non-null  int64  
 13  secu3        377638 non-null  int64  
 14  locp         377638 non-null  int64  
 15  actp         377638 non-null  object 
 16  etatp        377638 non-null  int64  
dtypes: float64(1), int64(12), object(4)
memory usage: 49.0+ MB


In [118]:
count_2nd_digit_1 = (df0521["usagers"]["secu"].astype(str).str[1] == "1").sum()
print (count_2nd_digit_1)
print (df0521["usagers"]["secu"].shape[0])
print (df0521["usagers"]["secu"].shape[0] - count_2nd_digit_1)

1612132
2509620
897488


ignore ci-dessus, c'était pour voir si le nombre de données de "secu" qui avaient un 1 en deuxième chiffre étaient l'écrasante majorité, mais ce n'est pas le cas. on perdrait trop de données à juste supprimer ceux qui ont un 2 ou un 3 en deuxième.

Regardons maintenant les données sur les lieux.

In [106]:
cols_recentes_lieux = set(dfs["lieux"].columns)
cols_anciennes_lieux = set(df0521["lieux"].columns)

unique_recent_lieux = cols_recentes_lieux-cols_anciennes_lieux 
unique_ancien_lieux = cols_anciennes_lieux-cols_recentes_lieux

print(unique_ancien_lieux,unique_recent_lieux)
print(cols_anciennes_lieux,cols_recentes_lieux)

{'env1', 'annee', 'num_acc', 'Unnamed: 0'} {'Num_Acc', 'year'}
{'voie', 'catr', 'lartpc', 'circ', 'env1', 'pr', 'prof', 'annee', 'situ', 'vosp', 'larrout', 'nbv', 'plan', 'v2', 'Unnamed: 0', 'surf', 'num_acc', 'pr1', 'vma', 'v1', 'infra'} {'year', 'voie', 'catr', 'lartpc', 'circ', 'pr', 'prof', 'situ', 'vosp', 'larrout', 'nbv', 'plan', 'v2', 'Num_Acc', 'surf', 'pr1', 'vma', 'v1', 'infra'}


Nous effectuons le même travail que précedemment: enlever "Unnamed:0", uniformiser "num_acc" et "Num_Acc" ainsi que "année" et "year". De plus, nous pouvons enlver le feature "Env1" qui exprime la proximité à une école, ce qui n'est pas pertinent pour le projet. 

In [107]:
df0521["lieux"]=df0521["lieux"].rename(columns={"annee":"year","num_acc":"Num_Acc"})
df0521["lieux"].drop(columns="env1",inplace=True)
df0521["lieux"].drop(columns="Unnamed: 0",inplace=True)



Maintenant, choisisson les features que nous souhaitons garder :
- "catr" (catégorie de route : autoroute, route nationale, etc...)
- "circ" (régime de circulation : sens unique, bidirectionnelle, etc...)
- "nbv" (nombre total de voies de la circulation)
- "vosp" (signalement de l'existence d'une voie réserve : piste cyclables, etc...)
- "prof" (déclivité de la route : plat, pente , etc...)
- "plan" (tracé en plan : recitiligne, courbé à gauche, etc...)
- "surf" (état de la surface : normale, mouillée, etc...)
- "infra" (aménagement : souterrain - tunnel, pont, etc...)

In [110]:
colonnes_interessantes = ["year", "Num_Acc", "catr", "circ", "nbv", "vosp", "prof", "plan", "surf", "infra"]
df0521_lieux_interessant = df0521["lieux"][colonnes_interessantes]
dfs_lieux_interessant = dfs["lieux"][colonnes_interessantes]

Puis rajoutons les à df_final2224 et df_final0521

In [112]:
KEY = ["year","Num_Acc"]

df_final2224 = dfs_lieux_interessant.merge(df_final2224,on=KEY,how="left") 
df_final0521 = df0521_lieux_interessant.merge(df_final0521, on=KEY, how="left")

In [113]:
df_final2224
df_final0521

,year,Num_Acc,catr,circ,nbv,vosp,prof,plan,surf,infra,...,agg,int,atm,col,com,adr,gps,lat,long,dep
0,2005,200500000001,3.0,2.0,2.0,0.0,1.0,1.0,1.0,0.0,...,2.0,1.0,1.0,3.0,11,CD41B,M,5051500.0,294400.0,590
1,2005,200500000001,3.0,2.0,2.0,0.0,1.0,1.0,1.0,0.0,...,2.0,1.0,1.0,3.0,11,CD41B,M,5051500.0,294400.0,590
2,2005,200500000001,3.0,2.0,2.0,0.0,1.0,1.0,1.0,0.0,...,2.0,1.0,1.0,3.0,11,CD41B,M,5051500.0,294400.0,590
3,2005,200500000001,3.0,2.0,2.0,0.0,1.0,1.0,1.0,0.0,...,2.0,1.0,1.0,3.0,11,CD41B,M,5051500.0,294400.0,590
4,2005,200500000001,3.0,2.0,2.0,0.0,1.0,1.0,1.0,0.0,...,2.0,1.0,1.0,3.0,11,CD41B,M,5051500.0,294400.0,590
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2224954,2019,201900058837,1.0,1.0,2.0,0.0,1.0,1.0,2.0,0.0,...,1.0,1.0,8.0,4.0,67482,Autoroute A.35,M,"48,5769000","7,7269000",67
2224955,2019,201900058838,1.0,3.0,3.0,0.0,1.0,1.0,1.0,0.0,...,1.0,1.0,1.0,6.0,94021,AUTOROUTE A6A,M,"48,7717000","2,3457600",94
2224956,2019,201900058839,1.0,1.0,1.0,0.0,1.0,3.0,1.0,0.0,...,1.0,1.0,1.0,7.0,78640,A86 EXT,M,"48,7772890","2,2237590",78
2224957,2019,201900058840,1.0,1.0,3.0,0.0,1.0,1.0,1.0,0.0,...,1.0,1.0,1.0,2.0,92047,A13,M,"48,8351236","2,1751101",92
